In [1]:
import pandas as pd

# Load the raw CSV with semicolon delimiter
file_path = 'data/Nigerian Stock Exchange All Share Index ASI 2012-2023.csv'
df = pd.read_csv(file_path, sep=';')

# 1. Dimensions (rows, columns)
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")

# 2. Exact column names (notice any trailing spaces)
print("Exact Column Names:")
print(df.columns.tolist(), "\n")

# 3. Schema & Data Types summary
print("Schema Information:")
df.info()

# 4. Preview the first 3 rows
df.head(3)

Shape: 2948 rows, 8 columns

Exact Column Names:
['Date', 'Closing Price', 'Opening Price', 'Highest Price of the Day', 'Lowest Price of the Day', 'Trading Vol.', 'Intra Day changes %', 'Daily Changes '] 

Schema Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2948 entries, 0 to 2947
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Date                      2948 non-null   object
 1   Closing Price             2948 non-null   object
 2   Opening Price             2948 non-null   object
 3   Highest Price of the Day  2948 non-null   object
 4   Lowest Price of the Day   2948 non-null   object
 5   Trading Vol.              2948 non-null   object
 6   Intra Day changes %       2948 non-null   object
 7   Daily Changes             2948 non-null   object
dtypes: object(8)
memory usage: 184.4+ KB


,Date,Closing Price,Opening Price,Highest Price of the Day,Lowest Price of the Day,Trading Vol.,Intra Day changes %,Daily Changes
0,22 12 2023,"? 74,023.27","? 74,288.88","? 74,430.47","? 73,993.49","? 423,200,000.00",-0.360%,0.000%
1,21 12 2023,"? 74,289.02","? 73,397.71","? 74,292.76","? 73,397.71","? 760,930,000.00",1.210%,0.359%
2,20 12 2023,"? 73,397.71","? 72,470.10","? 73,399.61","? 72,470.10","? 552,220,000.00",1.280%,-1.200%


# Data Cleaning Pipeline

In [2]:
# Standardize Column Names
# OBSERVATION: Column 'Daily Changes ' contains a trailing space, which causes
# KeyError when queried. Renaming to clean, snake_case names simplifies code.
# Strip any leading/trailing whitespace from column headers
df.columns = df.columns.str.strip()

# Rename to clean, standardized names for easier analysis
column_mapping = {
    'Date': 'date',
    'Closing Price': 'close',
    'Opening Price': 'open',
    'Highest Price of the Day': 'high',
    'Lowest Price of the Day': 'low',
    'Trading Vol.': 'volume',
    'Intra Day changes %': 'intraday_change_pct',
    'Daily Changes': 'daily_change_pct'
}
df = df.rename(columns=column_mapping)

# ------------------------------------------------------------------------------
# Clean Price & Volume Columns (Remove '?' and commas, convert to float)
# OBSERVATION: Values like '? 74,023.27' contain '?' (currency placeholder) 
# and ',' (thousands separators), causing pandas to read them as text strings (object).
# Mathematical operations, averages, and charts require numeric float types.
# ------------------------------------------------------------------------------
price_and_vol_cols = ['close', 'open', 'high', 'low', 'volume']

for col in price_and_vol_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('?', '', regex=False)    # Remove currency symbol artifact
        .str.replace(',', '', regex=False)    # Remove thousands separators
        .str.strip()                          # Remove surrounding spaces
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ------------------------------------------------------------------------------
# Clean Percentage Columns (Remove '%' and convert to float)
# OBSERVATION: Percentages include the literal '%' symbol (e.g., '-0.360%'),
# keeping them as strings. Removing '%' allows numeric calculations.
# ------------------------------------------------------------------------------
pct_cols = ['intraday_change_pct', 'daily_change_pct']

for col in pct_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace('%', '', regex=False)    # Remove percentage symbol
        .str.strip()
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ------------------------------------------------------------------------------
# Parse Dates into Datetime Format
# OBSERVATION: Dates are stored as text strings formatted as 'DD MM YYYY'
# (e.g., '22 12 2023'). Parsing into pandas datetime enables date-based slicing,
# resampling by month/year, and proper time-series indexing.
# ------------------------------------------------------------------------------
df['date'] = pd.to_datetime(df['date'], format='%d %m %Y')

# ------------------------------------------------------------------------------
# Chronological Sorting & Index Reset
# OBSERVATION: The raw data is sorted in reverse chronological order (2023 down to 2012).
# Time-series models, moving averages, and cumulative returns require chronological order.
# ------------------------------------------------------------------------------
df = df.sort_values(by='date', ascending=True).reset_index(drop=True)

# ------------------------------------------------------------------------------
# Verify Cleaned Dataset
# OBSERVATION: Confirms non-null counts, data types (float64 and datetime64),
# and inspects the first few rows of the cleaned data.
# ------------------------------------------------------------------------------
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2948 entries, 0 to 2947
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 2948 non-null   datetime64[ns]
 1   close                2948 non-null   float64       
 2   open                 2948 non-null   float64       
 3   high                 2948 non-null   float64       
 4   low                  2948 non-null   float64       
 5   volume               2948 non-null   float64       
 6   intraday_change_pct  2948 non-null   float64       
 7   daily_change_pct     2948 non-null   float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 184.4 KB


,date,close,open,high,low,volume,intraday_change_pct,daily_change_pct
0,2012-01-30,20731.72,20939.10,20950.02,20731.72,0.0,-0.77,-0.690
1,2012-01-31,20875.83,20818.56,21009.92,20789.48,0.0,0.70,0.409
2,2012-02-01,20790.88,20875.34,20941.03,20790.88,0.0,-0.41,-0.149
3,2012-02-02,20822.00,20872.94,20900.58,20785.40,0.0,0.15,-0.267
4,2012-02-03,20877.64,20822.00,20863.93,20786.35,0.0,0.27,0.417


In [3]:
# Optional: Impute the single 0.00 open on 2014-07-24 (equals 42,542.51)
df.loc[df['open'] == 0, 'open'] = df['close'] / (1 + df['intraday_change_pct'] / 100)

# Save Cleaned Dataset to Data Folder

In [4]:
# OBSERVATION: Saving with index=False prevents an unwanted 'Unnamed: 0' index column.
# Using standard comma separation allows any tool (Python, Excel, PowerBI, Tableau)
# to open it directly without needing sep=';'.
# ------------------------------------------------------------------------------
output_path = 'data/clean_Nigerian Stock Exchange All Share Index ASI 2012-2023.csv'
df.to_csv(output_path, index=False)

print(f"Cleaned dataset successfully saved to: {output_path}")

Cleaned dataset successfully saved to: data/clean_Nigerian Stock Exchange All Share Index ASI 2012-2023.csv


# Verify the Cleaned Schema & Data

In [5]:
# Verify data types and null counts
print("Cleaned Schema & Info:")
df.info()

print("\nMissing Values Count:")
print(df.isna().sum())

print("\nDate Range:")
print(f"From {df['date'].min().date()} to {df['date'].max().date()} ({df.shape[0]} trading days)")

# View first 5 rows (earliest dates from 2012)
df.head()

Cleaned Schema & Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2948 entries, 0 to 2947
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 2948 non-null   datetime64[ns]
 1   close                2948 non-null   float64       
 2   open                 2948 non-null   float64       
 3   high                 2948 non-null   float64       
 4   low                  2948 non-null   float64       
 5   volume               2948 non-null   float64       
 6   intraday_change_pct  2948 non-null   float64       
 7   daily_change_pct     2948 non-null   float64       
dtypes: datetime64[ns](1), float64(7)
memory usage: 184.4 KB

Missing Values Count:
date                   0
close                  0
open                   0
high                   0
low                    0
volume                 0
intraday_change_pct    0
daily_change_pct       0
dtype: int64

Dat

,date,close,open,high,low,volume,intraday_change_pct,daily_change_pct
0,2012-01-30,20731.72,20939.10,20950.02,20731.72,0.0,-0.77,-0.690
1,2012-01-31,20875.83,20818.56,21009.92,20789.48,0.0,0.70,0.409
2,2012-02-01,20790.88,20875.34,20941.03,20790.88,0.0,-0.41,-0.149
3,2012-02-02,20822.00,20872.94,20900.58,20785.40,0.0,0.15,-0.267
4,2012-02-03,20877.64,20822.00,20863.93,20786.35,0.0,0.27,0.417
